<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 2 (Martes) — Clasificación 1 — Árboles de decisión y bosques aleatorios**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# 🏀 Árbol vs. bosque aleatorio: ¿es un jugador superestrella?

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Clasificar jugadores como "superestrella" o no según su desempeño en la temporada y su carrera. |
| 🔍 Comprensión de los datos | Cargamos `Chapter11DataSet_Training.csv` (263 jugadores), exploramos variables con diagramas de caja por clase. |
| 🧹 Preparación de los datos | Convertimos `Team_Value` en binaria y dividimos entrenamiento/prueba (70/30). |
| 🤖 Modelado | Entrenamos un árbol de decisión y un bosque aleatorio (100 árboles) con las 16 variables. |
| 📊 Evaluación | Comparamos curvas ROC y matrices de confusión de ambos modelos. |
| 🚀 Despliegue y conclusiones | Concluimos cuál modelo conviene y por qué. |

> 🎯 **Analogía:** un cazatalentos que decide con reglas simples (árbol) frente a un comité de cazatalentos que votan (bosque).

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 1 | Lunes | Predicción: regresión lineal y series de tiempo |
| 👉 **2** | Martes | **Clasificación 1: árboles de decisión, bosques aleatorios y KNN** |
| 3 | Miércoles | Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales |
| 4 | Jueves | No supervisado: agrupación y asociación |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

### 📥 Datos del curso
Los archivos se descargan automáticamente (una sola vez) desde el repositorio del curso. Si ya los subió a `sample_data/`, no se vuelven a descargar.

In [ ]:
import os, urllib.request
REPO = 'https://gitlab.com/ejimenez/fiec_intro_machine_learning/-/raw/main'
os.makedirs('sample_data', exist_ok=True)
for nombre, dia in [('Chapter11DataSet_Training.csv', 'dia_3')]:
    ruta = f'sample_data/{nombre}'
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(f'{REPO}/{dia}/{nombre}', ruta)
        print('Descargado:', ruta)
    else:
        print('Ya existe:', ruta)

---
## 🎯 Fase 1 · Comprensión del negocio

Clasificar jugadores como "superestrella" o no según su desempeño en la temporada y su carrera.

📚 Lectura recomendada: [7]

## Análisis del Problema

El deporte, como inustria, mueve millones de dólares al año. Con respecto a las contrataciones, diferentes
tienen diferentes formas de “agrupar” o “catalogar” a los jugadores, pero en general hay una categoría de
jugadores sumamente cotizados que son considerados “súper estrellas”. Las implicaciones de estos jugadores
son muchas, pues no solo representan una gran inversión para el equipo sino que también se convierten en
una de las razones principales por las cuales el público va al estadio. A nivel de mercadotécnia, también es
una decsisión súmamente importante pues las súper estrellas suelen representar una de las principales cartas
comerciales.
Por estas razones, la decisión de si un jugador puede ser o no una súper estrella es de suma importancia, y
tener un modelo que pueda predecir potenciales súper estrellas para asisiter en la toma de estas decisiones
puede ser de suma importancia para diferentes equipos.

---
## 🔍 Fase 2 · Comprensión de los datos

Cargamos `Chapter11DataSet_Training.csv` (263 jugadores), exploramos variables con diagramas de caja por clase.

📚 Lectura recomendada: [10] [11]

## Entendimiento de los Datos

Para analizar este problema, se tiene un conjunto de datos con 263 observaciones y 19 variables:
- Nombre del jugador: cualitativa; con 263 valores diferentes.
- ID de la Posición: cualitativa; con 11 valores diferentes entre 0 y 11.
- Tiros: cuantitativa; cantidad de tiros en la temporada anterior, con un rango entre 19 y 687.
- Tiros Hechos: cuantitativa; cantidad de tiros anotados en la temporada anterior, con un rango entre 1
y 238.
- Puntos Personales: cuantitativa; puntos anotados personalmente por el atleta la temporada anterior,
con un rango entre 0 y 40.
- Puntos Totales: cuantitativa; cantidad de puntos que el atleta contribuyó a anotar, con un rango entre
0 y 130.
- Asistencias: cuantitativa; cantidad de veces que el atleta contribuyó a quitarle la posesión del balón al
rival, con un rango entre 0 y 121.
- Concesiones: cuantitativa; cantidad de veces que una jugada del atleta causó que el rival cediera una
ventaja a la ofensiva, con un valor entre 0 y 105.
- Bloqueos: cuantitativa; cantidad de veces que el atleta bloqueó a un rival la temporada anterior, con
un valor entre 0 y 1377.
- Asistencias a Bloqueos: cuantitativa; cantidad de veces que el atleta asistió a un compañero a bloquear
a un rival, con un rango entre 0 y 492.
- Faltas: cuantitativa; cantidad de veces que el atleta cometió una falta, con un rival entre 0 y 32.
- Años de profesional: cuantitativa; cantidad de año que el atleta ha jugado a nivel profesional, con un
rango entre 1 y 24.
- Tiros en la carrera: cuantitativa; cantidad de tiros en toda la carrera del atleta, con un rango de 19 a
14053.
- Tiros hechos en la carrera: cuantitativa; cantidad de tiros hechos en la carrera del atleta, con un rango
de 4 a 4256.
- Puntos Personales en la carrera: cuantitativa; cantidad de puntos personales en la carrera del atleta,
con un rango de 0 a 548.
- Puntos Totales en la carrera: cuantitativa; puntos totales del atleta en toda su carrera, con un rango de
2 a 2165.
- Asistencias en la carrera: cuantitativa; cantidad de asistencias en toda la carrera, con un rango de 3 a
1659.
- Concesiones en la carrera: cuantitativa; cantidad de concesiones en toda la carrera, con un rango de 1 a
1566.
- Valor para el equipo: cualitativa; valor que puede tener el jugador para el equipo, con cuatro valores
posibles: Jugador de Rol, Contribuidor, Jugador de Franquicia y Súper Estrella. Esta variable se
manipuló durante el análisis para convertirla en binaria, respondiendo a la pregunta: ¿Es el jugador
una súper estrella? donde 0 es ‘No’ y 1 es ‘Sí’.

---
## 🧹 Fase 3 · Preparación de los datos

Convertimos `Team_Value` en binaria y dividimos entrenamiento/prueba (70/30).

📚 Lectura recomendada: [7]

Antes de explorar los datos, es necesario cargarlos y aplicar la transformación respectiva a la variable “valor
para el equipo”, pues el problema consiste en determinar cuáles jugadores son potencialmente súper estrellas.
Adicionalmente, se procede a dividir el conjunto de datos en uno de entrenamiento y otro de prueba.

## librerías

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn.ensemble import RandomForestClassifier

from sklearn.tree import export_graphviz
import graphviz
import plotly.graph_objs as go
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

In [ ]:
# Cargar los datos desde el archivo CSV
datos = pd.read_csv('sample_data/Chapter11DataSet_Training.csv', header=0, na_values='?')

# Convertir la variable Team_Value en una variable binaria (0 o 1)
datos['Team_Value'] = [1 if x == 'Superstar' else 0 for x in datos['Team_Value']]

# Convertir la variable Team_Value en una variable categórica
datos['Team_Value'] = datos['Team_Value'].astype('category')

# Dividir los datos en conjuntos de entrenamiento y prueba
entrenamiento, prueba = train_test_split(datos, test_size=0.3, random_state=42)

In [ ]:
datos.shape

In [ ]:
# Mostrar un resumen estadístico
print(datos.describe())

Como podemos observar, la mayoría de las observaciones no son súper estrellas. Esto es esperado, pues la
esta categoría de jugadores generalmente son los menos.

In [ ]:
entrenamiento['Team_Value'].value_counts()

In [ ]:
# Contar las observaciones en cada categoría de la variable Team_Value
count = entrenamiento['Team_Value'].value_counts()

# Crear un diagrama de barras
plt.bar(count.index, count.values)

# Establecer el título y las etiquetas de los ejes
plt.title('Distribución de Valor para el Equipo')
plt.ylabel('Observaciones')
plt.xlabel('¿Es Súper Estrella? (1 = Sí)')

# Mostrar el diagrama de barras
plt.show()

---
## 🔍 Fase 2 · Comprensión de los datos — diagramas de caja por clase

Cargamos `Chapter11DataSet_Training.csv` (263 jugadores), exploramos variables con diagramas de caja por clase.

Para efectos de determinar qué variables influencian la determinación de si es una súper estrella o no, podemos
analizar algunas de las variales que acumulan el desempeño de la carrera.
En el caso de la variable Career_TP, se puede apreciar una diferencia considerable entre las dos categorías.
Sin embargo, los valores extremos de la categoría ‘No’ o 0 son mayores. Estos puntos anómalos en esa
categoría pueden ser los jugadores de franquicia.

In [ ]:
# Boxplot de Career_TP
plt.boxplot([entrenamiento[entrenamiento['Team_Value'] == 0]['Career_TP'],
             entrenamiento[entrenamiento['Team_Value'] == 1]['Career_TP']],
            tick_labels=['No', 'Sí'])
plt.title('Distribución de Career_TP por Valor para el Equipo')
plt.xlabel('¿Es Súper Estrella? (1 = Sí)')
plt.ylabel('Career_TP')
plt.show()

La variable Career_PP se comporta de manera similar:

In [ ]:
# Boxplot de Career_PP
plt.boxplot([entrenamiento[entrenamiento['Team_Value'] == 0]['Career_PP'],
             entrenamiento[entrenamiento['Team_Value'] == 1]['Career_PP']],
            tick_labels=['No', 'Sí'])
plt.title('Distribución de Career_PP por Valor para el Equipo')
plt.xlabel('¿Es Súper Estrella? (1 = Sí)')
plt.ylabel('Career_PP')
plt.show()

Y también la variable Career_Assists:

In [ ]:
# Boxplot de Career_Assists
plt.boxplot([entrenamiento[entrenamiento['Team_Value'] == 0]['Career_Assists'],
             entrenamiento[entrenamiento['Team_Value'] == 1]['Career_Assists']])
plt.xticks([1, 2], ['No', 'Sí'])
plt.xlabel('¿Es Súper Estrella?')
plt.ylabel('Career_Assists')
plt.title('Distribución de Career_Assists por Valor para el Equipo')
plt.show()

Cuando se analizan las variables a nivel de la temporada anterior, el patrón se mantiene, aunque las diferencias
son menores:

In [ ]:
# Boxplot de Total_Points
plt.boxplot([entrenamiento[entrenamiento['Team_Value'] == 0]['Total_Points'],
             entrenamiento[entrenamiento['Team_Value'] == 1]['Total_Points']])
plt.xticks([1, 2], ['No', 'Sí'])
plt.xlabel('¿Es Súper Estrella?')
plt.ylabel('Total_Points')
plt.title('Distribución de Total_Points por Valor para el Equipo')
plt.show()

La variable Career_PP se comporta de manera similar:

In [ ]:
# Boxplot de Personal_Points
plt.boxplot([entrenamiento[entrenamiento['Team_Value'] == 0]['Personal_Points'],
             entrenamiento[entrenamiento['Team_Value'] == 1]['Personal_Points']])
plt.xticks([1, 2], ['No', 'Sí'])
plt.xlabel('¿Es Súper Estrella?')
plt.ylabel('Personal_Points')
plt.title('Distribución de Personal_Points por Valor para el Equipo')
plt.show()




Y también la variable Career_Assists:

In [ ]:
# Crear boxplot para Assists
plt.boxplot([entrenamiento[entrenamiento['Team_Value'] == 0]['Assists'], entrenamiento[entrenamiento['Team_Value'] == 1]['Assists']], tick_labels=['No Superstar', 'Superstar'])
plt.title('Distribución de Assists por Valor para el Equipo')
plt.ylabel('Assists')
plt.xlabel('¿Es Súper Estrella? (1 = Sí)')
plt.show()

Hay algunas variables que tienden a favorecer el no, como por ejemplo la cantidad de faltas la temporada
anterior:

In [ ]:
plt.boxplot([entrenamiento[entrenamiento['Team_Value'] == 0]['Fouls'], entrenamiento[entrenamiento['Team_Value'] == 1]['Fouls']], tick_labels=['No Superstar', 'Superstar'])
plt.title('Distribución de Fouls por Valor para el Equipo')
plt.ylabel('Fouls')
plt.xlabel('¿Es Súper Estrella? (1 = Sí)')
plt.show()

A pesar de que el análisis exploratorio nos indica que podríamos guiarnos nada más por las estadísticas a nivel
de la carrera, si hacemos esto, corremos el riesgo de no tomar encuenta súper estrellas potenciales que están
en etapas tempranas en su carrera. Para evitar esto, y para aprovechar también la capacidad de las redes
neuronales, vamos a tomar en cuenta todas las variables cuantitativas para intentar responder la pregunta.

---
## 🤖 Fase 4 · Modelado

Entrenamos un árbol de decisión y un bosque aleatorio (100 árboles) con las 16 variables.

📚 Lectura recomendada: [3] [4]

Se crea el árbol de decisión:

In [ ]:
# Crear el modelo de árbol de decisión
modelo_arbol = DecisionTreeClassifier()

# Seleccionar las variables predictoras del conjunto de entrenamiento

feature_names = ['Shots', 'Makes', 'Personal_Points',
       'Total_Points', 'Assists', 'Concessions', 'Blocks', 'Block_Assists',
       'Fouls', 'Years_Pro', 'Career_Shots', 'Career_Makes', 'Career_PP',
       'Career_TP', 'Career_Assists', 'Career_Con']

# Seleccionar las variables predictoras del conjunto de entrenamiento
X_entrenamiento = entrenamiento[feature_names]

# Seleccionar la variable de respuesta del conjunto de entrenamiento
y_entrenamiento = entrenamiento['Team_Value']

modelo_arbol.fit(X_entrenamiento, y_entrenamiento)

In [ ]:
# Trazar el árbol de decisión
dot_data = export_graphviz(modelo_arbol, out_file=None,
                           feature_names=feature_names,
                           class_names=['No', 'Yes'],
                           filled=True, rounded=True,
                           special_characters=True)
graph = graphviz.Source(dot_data)
graph.render('arbol')
graph

In [ ]:
# Seleccionar las variables predictoras del conjunto de prueba

feature_names = ['Shots', 'Makes', 'Personal_Points',
       'Total_Points', 'Assists', 'Concessions', 'Blocks', 'Block_Assists',
       'Fouls', 'Years_Pro', 'Career_Shots', 'Career_Makes', 'Career_PP',
       'Career_TP', 'Career_Assists', 'Career_Con']


X_prueba = prueba[feature_names]

# Hacer predicciones con el conjunto de datos de prueba utilizando el modelo de árbol de decisión
prediccionesAD = modelo_arbol.predict(X_prueba)
# Imprime las predicciones
print(prediccionesAD)

Finalmente, el **bosque aleatorio**:

In [ ]:
#crear modelo

# Seleccionar las variables predictoras del conjunto de entrenamiento
feature_names = ['Shots', 'Makes', 'Personal_Points',
       'Total_Points', 'Assists', 'Concessions', 'Blocks', 'Block_Assists',
       'Fouls', 'Years_Pro', 'Career_Shots', 'Career_Makes', 'Career_PP',
       'Career_TP', 'Career_Assists', 'Career_Con']

# Seleccionar las variables predictoras del conjunto de entrenamiento
X_entrenamiento = entrenamiento[feature_names]

# Seleccionar la variable de respuesta del conjunto de entrenamiento
y_entrenamiento = entrenamiento['Team_Value']

# Crear el modelo de bosque aleatorio
modelo_bosque = RandomForestClassifier(n_estimators=100, random_state=14234)
modelo_bosque.fit(X_entrenamiento, y_entrenamiento)


In [ ]:
# Imprimir el primer árbol en el bosque
dot_data = export_graphviz(modelo_bosque.estimators_[0], out_file=None,
                           feature_names=feature_names,
                           class_names=['No', 'Yes'],
                           filled=True, rounded=True,
                           special_characters=True)
graph = graphviz.Source(dot_data)
graph.render('arbol_bosque')
graph

In [ ]:
X_entrenamiento.shape

In [ ]:
# Visualizar la importancia de cada variable en la clasificación
# Obtener la importancia de cada variable
importancias = modelo_arbol.feature_importances_

# Crear un DataFrame con los nombres de las variables y sus respectivas importancias
df_importancias = pd.DataFrame({'Variable': feature_names, 'Importancia': importancias})

# Ordenar el DataFrame de forma descendente según la importancia de las variables
df_importancias = df_importancias.sort_values('Importancia', ascending=False)

# Imprimir el DataFrame resultante
print(df_importancias)


# Creamos una lista con los nombres de las variables y sus importancias
var_imp = list(zip(X_entrenamiento[feature_names].columns[:-1], modelo_arbol.feature_importances_))
# Ordenamos la lista por importancia en orden ascendente
var_imp.sort(key=lambda x: x[1])

# Creamos un objeto de barras verticales
fig = go.Figure(data=[go.Bar(x=[x[1] for x in var_imp], y=[x[0] for x in var_imp], orientation='h')])

# Definimos el título del gráfico y los nombres de los ejes
fig.update_layout(title='Importancia de las variables', xaxis_title='Importancia', yaxis_title='Variable')

# Mostramos el gráfico
fig.show()

In [ ]:
# Realiza predicciones sobre los datos de prueba
prediccionesRF = modelo_bosque.predict(prueba[feature_names])

# Imprime las predicciones
print(prediccionesRF)

---
## 📊 Fase 5 · Evaluación

Comparamos curvas ROC y matrices de confusión de ambos modelos.

📚 Lectura recomendada: [5] [6]

Averiguamos el desempeño del árbol:

**Arbol de decision**

![Curvas ROC](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch06/images/06_10.png)
*Figura: ejemplo de curvas ROC y su AUC. [9]*

In [ ]:
# Convierte las predicciones a valores numéricos
predicciones = modelo_arbol.predict_proba(prueba[feature_names])[:, 1]   # probabilidades, no etiquetas

# Calcula la tasa de verdaderos positivos, la tasa de falsos positivos y los umbrales utilizando la función roc_curve
fpr, tpr, umbrales = roc_curve(prueba['Team_Value'], predicciones)

# Calcula el valor del área bajo la curva ROC (AUC)
auc_score = auc(fpr, tpr)
print("Score ",auc_score)

# Dibuja la curva ROC del modelo
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='Curva ROC (AUC = %0.2f)' % auc_score)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos')
plt.title('Curva ROC del modelo Árbol')
plt.legend(loc="lower right")
plt.show()

**Bosque aleatorio**

In [ ]:
# Convierte las predicciones a valores numéricos
predicciones = modelo_bosque.predict_proba(prueba[feature_names])[:, 1]

# Calcula la tasa de verdaderos positivos, la tasa de falsos positivos y los umbrales utilizando la función roc_curve
fpr, tpr, umbrales = roc_curve(prueba['Team_Value'], predicciones)

# Calcula el valor del área bajo la curva ROC (AUC)
auc_score = auc(fpr, tpr)
print("Score ",auc_score)

# Dibuja la curva ROC del modelo
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='Curva ROC (AUC = %0.2f)' % auc_score)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos')
plt.title('Curva ROC del bosque aleatorio')
plt.legend(loc="lower right")
plt.show()

![Matriz de confusión](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch06/images/06_08.png)
*Figura: cómo leer una matriz de confusión (VP, FN, FP, VN). [9]*

**Matriz de confusion**


### **Arbol de decisión**



In [ ]:
# Calcula la matriz de confusión utilizando la función confusion_matrix
matriz_confusion = confusion_matrix(prueba['Team_Value'], prediccionesAD)

# Imprime la matriz de confusión
print(matriz_confusion)

In [ ]:
# Crea una instancia de la clase ConfusionMatrixDisplay
cm_display = ConfusionMatrixDisplay(matriz_confusion, display_labels=['0', '1'])

# Genera la gráfica de la matriz de confusión
fig, ax = plt.subplots(figsize=(8, 8))
cm_display.plot(ax=ax)
plt.show()



---



\begin{equation}Exactitud=\frac{VP+VN}{Total}\end{equation}

In [ ]:
print("Precisión global:", (matriz_confusion[0, 0] + matriz_confusion[1, 1]) / sum(sum(matriz_confusion)))

\begin{equation}\text{Sensibilidad}=\frac{VP}{\text{Total Positivos}}\end{equation}

In [ ]:
print("Sensibilidad:", matriz_confusion[1, 1] / (matriz_confusion[1, 1] + matriz_confusion[1, 0]))

\begin{equation}\text{Especificidad}=\frac{VN}{\text{Total Negativos}}\end{equation}

In [ ]:
print("Especificidad:", matriz_confusion[0, 0] / (matriz_confusion[0, 0] + matriz_confusion[0, 1]))

**Bosque aleatorio**

In [ ]:
# Calcula la matriz de confusión utilizando la función confusion_matrix
matriz_confusion = confusion_matrix(prueba['Team_Value'], prediccionesRF)

# Imprime la matriz de confusión
print(matriz_confusion)

In [ ]:
# Crea una instancia de la clase ConfusionMatrixDisplay
cm_display = ConfusionMatrixDisplay(matriz_confusion, display_labels=['0', '1'])

# Genera la gráfica de la matriz de confusión
fig, ax = plt.subplots(figsize=(8, 8))
cm_display.plot(ax=ax)
plt.show()



---



\begin{equation}Exactitud=\frac{VP+VN}{Total}\end{equation}

In [ ]:
print("Precisión global:", (matriz_confusion[0, 0] + matriz_confusion[1, 1]) / sum(sum(matriz_confusion)))

\begin{equation}\text{Sensibilidad}=\frac{VP}{\text{Total Positivos}}\end{equation}

In [ ]:
print("Sensibilidad:", matriz_confusion[1, 1] / (matriz_confusion[1, 1] + matriz_confusion[1, 0]))

\begin{equation}\text{Especificidad}=\frac{VN}{\text{Total Negativos}}\end{equation}

In [ ]:
print("Especificidad:", matriz_confusion[0, 0] / (matriz_confusion[0, 0] + matriz_confusion[0, 1]))

---
## 🚀 Fase 6 · Despliegue y conclusiones

Concluimos cuál modelo conviene y por qué.

📚 Lectura recomendada: [7]

#**Conclusiones **

**Sus conclusiones finales son?**

---
## 🧪 Refuerzo: precisión, recall y F1; importancia de variables

La exactitud no cuenta toda la historia. **Precisión** = de los que predije "superestrella", ¿cuántos lo son? **Recall** = de las superestrellas reales, ¿cuántas encontré? **F1** es su media armónica [6]. El bosque aleatorio además estima la **importancia** de cada variable [4].

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report

pred_ad = modelo_arbol.predict(prueba[feature_names])
pred_rf = modelo_bosque.predict(prueba[feature_names])
print('=== Árbol de decisión ==='); print(classification_report(prueba['Team_Value'], pred_ad, zero_division=0))
print('=== Bosque aleatorio ==='); print(classification_report(prueba['Team_Value'], pred_rf, zero_division=0))
print(pd.Series(modelo_bosque.feature_importances_, index=feature_names).sort_values(ascending=False).head(6).round(3))

### ✍️ Ejercicios
1. ¿Cuál modelo tiene mejor *recall* para la clase "Superestrella"?
2. Entrene el bosque solo con las 6 variables más importantes: ¿pierde mucho desempeño?

---
## ✅ Conclusiones y siguientes pasos

El bosque suele superar al árbol individual porque promedia muchos árboles distintos [4]. **Corrección importante en esta versión:** antes la evaluación del "bosque" usaba por error las predicciones del árbol; y las curvas ROC se calculaban con etiquetas 0/1 en lugar de probabilidades, lo que reduce la curva a un solo punto [5].

---
## 📚 Referencias

Fuentes adicionales para profundizar: [8].

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** Breiman, L., Friedman, J. H., Olshen, R. A., & Stone, C. J. (1984). *Classification and Regression Trees*. Wadsworth.

**[4]** Breiman, L. (2001). Random forests. *Machine Learning, 45*(1), 5–32. https://doi.org/10.1023/A:1010933404324

**[5]** Fawcett, T. (2006). An introduction to ROC analysis. *Pattern Recognition Letters, 27*(8), 861–874. https://doi.org/10.1016/j.patrec.2005.10.010

**[6]** Powers, D. M. W. (2011). Evaluation: From precision, recall and F-measure to ROC, informedness, markedness and correlation. *Journal of Machine Learning Technologies, 2*(1), 37–63.

**[7]** James, G., Witten, D., Hastie, T., Tibshirani, R., & Taylor, J. (2023). *An Introduction to Statistical Learning with Applications in Python*. Springer. https://doi.org/10.1007/978-3-031-38747-0

**[8]** Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … Duchesnay, E. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html

**[9]** Raschka, S., & Mirjalili, V. (2019). *Python Machine Learning* (3.ª ed.). Packt Publishing. (Figuras usadas desde https://github.com/rasbt/python-machine-learning-book-3rd-edition)

**[10]** McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 56–61. https://doi.org/10.25080/Majora-92bf1922-00a

**[11]** Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering, 9*(3), 90–95. https://doi.org/10.1109/MCSE.2007.55